### 01 — pybaseball_installation_check

Purpose: Confirm `pybaseball` can pull Statcast and Fangraph data end-to-end against this environment.

Caching is enabled and pointed at `../data/cache/` so re-running this notebook is instant after the first pull.

In [ ]:
from pathlib import Path

import pandas as pd
import pybaseball
from pybaseball import cache, statcast

DATA_DIR = Path("..") / "data" / "cache"
DATA_DIR.mkdir(parents=True, exist_ok=True)
cache.config.cache_directory = str(DATA_DIR.resolve())
cache.config.save()
cache.enable()

print("pybaseball:", pybaseball.__version__)
print("pandas    :", pd.__version__)
print("cache dir :", cache.config.cache_directory)

In [ ]:
df = statcast("2024-06-15", "2024-06-15")
df.shape

In [ ]:
print("rows, cols:", df.shape)
print("\ndtype counts:")
print(df.dtypes.value_counts())
df.head(3)

In [ ]:
print(f"{len(df.columns)} columns:")
for c in df.columns:
    print(f"  {c}")

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss[miss > 0].to_frame("pct_missing")

In [ ]:
print("pitch_type missingness:", df["pitch_type"].isna().mean())
print("pitch_name missingness:", df["pitch_name"].isna().mean())
print("\npitch_type value counts:")
print(df["pitch_type"].value_counts(dropna=False))
print("\npitch_name value counts:")
print(df["pitch_name"].value_counts(dropna=False))

---
### Player-Specific Feature Enrichment

The goal is to enrich each pitch row with **prior-year** pitcher and batter arsenal stats (usage + per-pitch-type outcomes).

| Table | pybaseball function | Why |
|---|---|---|
| Pitcher arsenal | `statcast_pitcher_arsenal_stats(year)` | `pitch_usage` + outcome stats per pitch type |
| Batter vs pitch types | `statcast_batter_pitch_arsenal(year)` | `bat_pitch_usage` + outcome stats per pitch type |

Enrichment is applied via shared `enrich_statcast()` from `utils.features.enrichment`.

In [ ]:
# ── Pitcher pitch-usage % (prior year) ────────────────────────────────────────
from pybaseball import statcast_pitcher_pitch_arsenal, statcast_pitcher_arsenal_stats

SEASON = int(df["game_year"].max())   # e.g. 2024
PRIOR  = SEASON - 1                   # e.g. 2023  ← leak-free

pit_usage = statcast_pitcher_pitch_arsenal(PRIOR, arsenal_type="n_")
print("pit_usage shape:", pit_usage.shape)
print("\nColumns:", pit_usage.columns.tolist())
pit_usage.head(3)

In [ ]:
# Pitcher per-pitch outcome stats (prior year)
pit_outcomes = statcast_pitcher_arsenal_stats(PRIOR, minPA=25)
print("pit_outcomes shape:", pit_outcomes.shape)
print("\nColumns:", pit_outcomes.columns.tolist())
pit_outcomes.head(6)

In [ ]:
# batter expected stats (prior year)
from pybaseball import statcast_batter_expected_stats, statcast_batter_pitch_arsenal

bat_xstats = statcast_batter_expected_stats(PRIOR)
print("bat_xstats shape:", bat_xstats.shape)
print("\nColumns:", bat_xstats.columns.tolist())
bat_xstats.head(3)

In [ ]:
# Batter pitch-type vulnerability (prior year)
bat_vs_pitch = statcast_batter_pitch_arsenal(PRIOR, minPA=25)
print("bat_vs_pitch shape:", bat_vs_pitch.shape)
print("\nColumns:", bat_vs_pitch.columns.tolist())
bat_vs_pitch.head(6)

In [ ]:
# Join all prior-year stats to the Statcast frame ───────────────────────────
pit_usage_w = pit_usage.rename(columns={"pitcher_id": "pitcher"})
pit_out_id_cols   = ["player_id", "last_name", "first_name", "year"]
pit_out_stat_cols = [c for c in pit_outcomes.columns if c not in pit_out_id_cols + ["pitch_type", "pitch_name"]]

pit_outcomes_w = (
    pit_outcomes
    .pivot_table(
        index="player_id",
        columns="pitch_type",
        values=pit_out_stat_cols,
        aggfunc="first",
    )
)
# flatten multi-index
pit_outcomes_w.columns = [f"{stat}_{pt}" for stat, pt in pit_outcomes_w.columns]

pit_outcomes_w = pit_outcomes_w.reset_index().rename(columns={"player_id": "pitcher"})
bat_xstats_w = bat_xstats.rename(columns={"player_id": "batter"})
bat_vs_id_cols   = ["player_id", "last_name", "first_name", "year"]
bat_vs_stat_cols = [c for c in bat_vs_pitch.columns if c not in bat_vs_id_cols + ["pitch_type", "pitch_name"]]

bat_vs_pitch_w = (
    bat_vs_pitch
    .pivot_table(
        index="player_id",
        columns="pitch_type",
        values=bat_vs_stat_cols,
        aggfunc="first",
    )
)
bat_vs_pitch_w.columns = [f"bat_{stat}_{pt}" for stat, pt in bat_vs_pitch_w.columns]
bat_vs_pitch_w = bat_vs_pitch_w.reset_index().rename(columns={"player_id": "batter"})

# merge to pitch level
df_enr = (
    df
    .merge(pit_usage_w,    on="pitcher", how="left", suffixes=("", "_pu"))
    .merge(pit_outcomes_w, on="pitcher", how="left", suffixes=("", "_po"))
    .merge(bat_xstats_w,   on="batter",  how="left", suffixes=("", "_bx"))
    .merge(bat_vs_pitch_w, on="batter",  how="left", suffixes=("", "_bv"))
)

print("Original shape :", df.shape)
print("Enriched shape :", df_enr.shape)

n_ff_col   = "n_ff"   if "n_ff"   in df_enr.columns else None
xwoba_col  = "xwoba"  if "xwoba"  in df_enr.columns else None
if n_ff_col:
    print(f"Pitcher arsenal coverage (n_ff non-null): "
          f"{df_enr[n_ff_col].notna().mean():.1%}")
if xwoba_col:
    print(f"Batter xwOBA coverage (xwoba non-null)  : "
          f"{df_enr[xwoba_col].notna().mean():.1%}")

preview_cols = (
    ["pitcher", "p_throws"]
    + [c for c in ["n_ff", "n_si", "n_sl", "n_ch", "n_cu"] if c in df_enr.columns]
    + ["batter", "stand"]
    + [c for c in ["xwoba", "xba"] if c in df_enr.columns]
)
df_enr[preview_cols].head(5)